In [2]:
import os, sys
from pathlib import Path
os.chdir(Path(r"C:\Users\amand\OneDrive - Fisk University\Documents\ResumeBiasResearch"))
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from fairlearn.postprocessing import ThresholdOptimizer

from src.fairness.group_definition import get_gender_groups, get_ethnicity_groups, print_group_diagnostics
from src.fairness.metrics import compute_all_fairness_metrics as compute_all

# Load data
data = np.load('data/processed/prepared_data.npz')
X_train, X_test = data['X_train'], data['X_test']
y_train_gender = data['y_train_gender']
y_test_gender = data['y_test_gender']

raw = np.load('data/raw/FairCVdb.npy', allow_pickle=True).item()
gender_train = get_gender_groups(raw['Profiles Train'])
gender_test = get_gender_groups(raw['Profiles Test'])

In [6]:
# 1. Train a baseline model
base_clf = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
base_clf.fit(X_train, y_train_gender)

# Baseline fairness
y_pred_base = base_clf.predict(X_test)
fair_base = compute_all(y_test_gender, y_pred_base, gender_test)
print("--- Baseline Fairness ---")
print(f"DPD: {fair_base['demographic_parity_difference']:.3f}")
print(f"EOD: {fair_base['equal_opportunity_difference']:.3f}")
print(f"Min DIR: {min(fair_base['disparate_impact_ratios'].values()):.3f}")

# 2. Apply ThresholdOptimizer
mitigator = ThresholdOptimizer(
    estimator=base_clf,
    constraints="equalized_odds", 
    prefit=True, 
    predict_method='predict_proba'
)
mitigator.fit(X_train, y_train_gender, sensitive_features=gender_train)

# 3. Predict and measure fairness
y_pred_mitigated = mitigator.predict(X_test, sensitive_features=gender_test)
fair_mitigated = compute_all(y_test_gender, y_pred_mitigated, gender_test)

print("\n--- ThresholdOptimizer Fairness ---")
print(f"DPD: {fair_mitigated['demographic_parity_difference']:.3f}")
print(f"EOD: {fair_mitigated['equal_opportunity_difference']:.3f}")
print(f"Min DIR: {min(fair_mitigated['disparate_impact_ratios'].values()):.3f}")

Using '0' as reference (highest selection rate)
--- Baseline Fairness ---
DPD: 0.306
EOD: 0.081
Min DIR: 0.512
Using '0' as reference (highest selection rate)

--- ThresholdOptimizer Fairness ---
DPD: 0.263
EOD: 0.022
Min DIR: 0.570


In [7]:
import joblib

# Load tuned models
rf_tuned = joblib.load('results/models/random_forest_tuned.pkl')
xgb_tuned = joblib.load('results/models/xgboost_tuned.pkl')

models_to_test = {
    'Logistic Regression (Base)': base_clf,
    'Random Forest (Tuned)': rf_tuned,
    'XGBoost (Tuned)': xgb_tuned
}

results = []
for name, model in models_to_test.items():
    y_pred = model.predict(X_test)
    fair = compute_all(y_test_gender, y_pred, gender_test)
    results.append({
        'Model': name,
        'DPD': round(fair['demographic_parity_difference'], 3),
        'EOD': round(fair['equal_opportunity_difference'], 3),
        'Min DIR': round(min(fair['disparate_impact_ratios'].values()), 3)
    })

df_fairness_tuned = pd.DataFrame(results)
print(df_fairness_tuned.to_string(index=False))
df_fairness_tuned.to_csv('results/tables/fairness_tuned_models.csv', index=False)

Using '0' as reference (highest selection rate)
Using '0' as reference (highest selection rate)
Using '0' as reference (highest selection rate)
                     Model   DPD   EOD  Min DIR
Logistic Regression (Base) 0.306 0.081    0.512
     Random Forest (Tuned) 0.020 0.147    0.963
           XGBoost (Tuned) 0.002 0.231    0.996


In [8]:
# The first 2 columns of the 93-feature matrix are the encoded gender/ethnicity columns
# (We confirmed this in the SHAP analysis).
X_train_dropped = np.delete(X_train, [0, 1], axis=1)
X_test_dropped  = np.delete(X_test, [0, 1], axis=1)

# Retrain
lr_unaware = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
lr_unaware.fit(X_train_dropped, y_train_gender)
y_pred_unaware = lr_unaware.predict(X_test_dropped)

fair_unaware = compute_all(y_test_gender, y_pred_unaware, gender_test)
print("--- Fairness After Dropping Demographic Features ---")
print(f"DPD: {fair_unaware['demographic_parity_difference']:.3f}")
print(f"Min DIR: {min(fair_unaware['disparate_impact_ratios'].values()):.3f}")

Using '0' as reference (highest selection rate)
--- Fairness After Dropping Demographic Features ---
DPD: 0.218
Min DIR: 0.639


In [9]:
import pandas as pd

mitigation_results = [
    {'Method': 'Baseline (LR)', 'DPD': 0.306, 'EOD': 0.081, 'Min DIR': 0.512},
    {'Method': 'ThresholdOptimizer', 'DPD': 0.269, 'EOD': 0.028, 'Min DIR': 0.564},
    {'Method': 'Drop Demographics', 'DPD': 0.218, 'EOD': None, 'Min DIR': 0.639}
]

pd.DataFrame(mitigation_results).to_csv('results/tables/mitigation_comparison.csv', index=False)

In [10]:
# Compute the missing EOD for the Drop-Demographics test
from src.fairness.metrics import compute_all_fairness_metrics

fair_unaware_full = compute_all_fairness_metrics(y_test_gender, y_pred_unaware, gender_test)
eod_unaware = fair_unaware_full['equal_opportunity_difference']

print(f"Drop Demographics EOD: {eod_unaware:.3f}")

Using '0' as reference (highest selection rate)
Drop Demographics EOD: 0.060


In [11]:
import pandas as pd

# 1. Load the existing CSV
csv_path = 'results/tables/mitigation_comparison.csv'
df = pd.read_csv(csv_path)

# 2. Locate the row and update the EOD value
# We use .loc to find the row where the Method is 'Drop Demographics'
df.loc[df['Method'] == 'Drop Demographics', 'EOD'] = round(eod_unaware, 3)

# 3. Save it back to the same path
df.to_csv(csv_path, index=False)

print("Successfully updated the CSV! Here is the new table:")
print(df.to_string(index=False))

Successfully updated the CSV! Here is the new table:
            Method   DPD   EOD  Min DIR
     Baseline (LR) 0.306 0.081    0.512
ThresholdOptimizer 0.269 0.028    0.564
 Drop Demographics 0.218 0.060    0.639
